# Case Study 3: DEL regression pipeline
Load DEL trisynthons, run unified denoise (Poisson), then Featurize → Split → Train (RF+ECFP) → Evaluate → Infer.

In [ ]:
import uuid
import time
from pathlib import Path

import pandas as pd

from pyds import (
    Settings, Data, BaseClient, DelDenoise,
    Featurize, TVTSplit, Train, Evaluate, Infer
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [2]:
BASE_URL = "http://deepchem-server"
PROFILE = "del_profile"
PROJECT = "del_project"

repo_root = Path(".").resolve().parent
DATASET_PATH = (
    repo_root / "deepchem_server" / "core" / "tests" / "assets" /
    "DDR1_KINDEL_11111_sampled.parquet"
)

USE_DEMO_SAMPLE = True

run_id = uuid.uuid4().hex[:8]
print(f"Run ID: {run_id} — {time.strftime('%Y-%m-%d %H:%M:%S')}")

Run ID: e50147e2 — 2026-04-14 00:23:00


In [3]:
settings = Settings(profile=PROFILE, project=PROJECT, base_url=BASE_URL)

base_client = BaseClient(settings=settings)
data_client = Data(settings=settings)
del_denoise_client = DelDenoise(settings=settings)
featurize_client = Featurize(settings=settings)
split_client = TVTSplit(settings=settings)
train_client = Train(settings=settings)
evaluate_client = Evaluate(settings=settings)
infer_client = Infer(settings=settings)

print("healthcheck:", base_client.healthcheck())


healthcheck: {'status': 'ok'}


## 1. Prepare data
Drop missing SMILES and duplicates.

In [4]:
df = pd.read_parquet(DATASET_PATH)

base_model_df = (
    df.dropna(subset=["smiles"])
    .drop_duplicates(subset=["smiles"])
    .copy()
)

if USE_DEMO_SAMPLE:
    sorted_df = base_model_df.sort_values('target_enrichment', ascending=False).reset_index(drop=True)

    sampled_max = sorted_df.iloc[0:1].copy()
    sampled_max["tier"] = "max"
    
    sampled_veryhigh = sorted_df.iloc[1:100].sample(10, random_state=42)
    sampled_veryhigh["tier"] = "very_high"
    
    sampled_high = sorted_df.iloc[100:2000].sample(100, random_state=42)
    sampled_high["tier"] = "high"
    
    slice_mid = sorted_df.iloc[2000:6000]
    sampled_mid = slice_mid.sample(min(1000, len(slice_mid)), random_state=42)
    sampled_mid["tier"] = "mid"
    
    slice_low = sorted_df.iloc[6000:]
    sampled_low = slice_low.sample(min(2000, len(slice_low)), random_state=42)
    sampled_low["tier"] = "low"
    
    df_model = pd.concat(
        [sampled_max, sampled_veryhigh, sampled_high, sampled_mid, sampled_low], 
        ignore_index=True
    )
    
    df_model = df_model.sample(frac=1, random_state=42).reset_index(drop=True)

else:
    base_model_df["tier"] = "full_dataset"
    df_model = base_model_df.reset_index(drop=True)

print(df_model.shape, "cols:", list(df_model.columns)[:5], "...")

(3111, 14) cols: ['smiles', 'molecule_hash', 'smiles_a', 'smiles_b', 'smiles_c'] ...


## 2. Upload modeling table

In [5]:
temp_path = Path(f"/tmp/del_model_input_{run_id}.csv")
df_model.to_csv(temp_path, index=False)

upload_result = data_client.upload_data(
    file_path=temp_path,
    filename=f"del_model_input_{run_id}.csv",
    description=f"DDR1 DEL modeling table (run {run_id})",
)
dataset_address = upload_result["dataset_address"]
print("dataset_address:", dataset_address)


dataset_address: deepchem://del_profile/del_project/del_model_input_e50147e2.csv


## 3. Unified denoise and regression CSV
Unified Poisson enrichment for regression. Then upload features+label for ML.

In [6]:
denoise_result = del_denoise_client.run(
    dataset_address=dataset_address,
    output_key=f"del_denoised_{run_id}",
    strategy="unified",
    control_cols=["seq_matrix_1", "seq_matrix_2", "seq_matrix_3"],
    target_cols=["seq_target_1", "seq_target_2", "seq_target_3"],
    add_hit_labels=False,
    hit_percentile=90.0,
    alpha=0.05,
    drop_duplicates=True,
    use_disynthon_pairs=False,
)
denoised_address = denoise_result["denoised_dataset_address"]
print("denoised:", denoised_address)

df_denoised = data_client.get(denoised_address)

regression_temp_path = Path(f"/tmp/del_regression_{run_id}.csv")
df_denoised.to_csv(regression_temp_path, index=False)
regression_upload = data_client.upload_data(
    file_path=regression_temp_path,
    filename=f"del_regression_{run_id}.csv",
    description=f"DDR1 regression run {run_id}",
)
regression_dataset_address = regression_upload["dataset_address"]
print("regression_dataset_address:", regression_dataset_address)


denoised: deepchem://del_profile/del_project/del_denoised_e50147e2.csv
regression_dataset_address: deepchem://del_profile/del_project/del_regression_e50147e2.csv


## 4. Featurize (ECFP)

In [7]:
featurize_result = featurize_client.run(
    dataset_address=regression_dataset_address,
    featurizer="ecfp",
    output=f"del_ecfp_{run_id}",
    dataset_column="smiles",
    label_column="Poisson_Enrichment",
    feat_kwargs={"radius": 2, "size": 2048},
)
featurized_address = featurize_result["featurized_file_address"]
print("featurized_address:", featurized_address)

featurized_address: deepchem://del_profile/del_project/del_ecfp_e50147e2


## 5. Split

In [8]:
split_result = split_client.run(
    splitter_type="random",
    dataset_address=featurized_address,
    frac_train=0.7,
    frac_valid=0.15,
    frac_test=0.15,
)
train_addr, valid_addr, test_addr = split_result["train_valid_test_split_results_address"]
print("train:", train_addr)
print("valid:", valid_addr)
print("test: ", test_addr)


train: deepchem://del_profile/del_project/del_ecfp_e50147e2_train
valid: deepchem://del_profile/del_project/del_ecfp_e50147e2_valid
test:  deepchem://del_profile/del_project/del_ecfp_e50147e2_test


## 6. Train (random forest)

In [ ]:
train_result = train_client.run(
    dataset_address=train_addr,
    model_type="random_forest_regressor",
    model_name=f"rf_del_enrichment_{run_id}",
    init_kwargs={
        "n_jobs": -1,
    },
    train_kwargs={},
)
model_address = train_result["trained_model_address"]
print("model_address:", model_address)

model_address: deepchem://del_profile/del_project/rf_del_enrichment_e50147e2


## 7. Evaluate

In [ ]:
evaluate_result = evaluate_client.run(
    dataset_addresses=[valid_addr, test_addr],
    model_address=model_address,
    metrics=["pearson_r2_score", "rms_score", "mae_error"],
    output_key=f"eval_del_{run_id}",
    is_metric_plots=False,
)
evaluation_address = evaluate_result["evaluation_result_address"]
eval_results = data_client.get(evaluation_address)
print("evaluation:", evaluation_address)
for split_name, split_addr in [("Valid", valid_addr), ("Test", test_addr)]:
    print(f"\n{split_name}")
    for metric, value in eval_results[split_addr].items():
        print(f"  {metric}: {value:.4f}")


evaluation: deepchem://del_profile/del_project/eval_del_e50147e2.json

Valid
  pearson_r2_score: 0.0157
  rms_score: 0.8225
  mae_score: 0.2399

Test
  pearson_r2_score: 0.0646
  rms_score: 0.7792
  mae_score: 0.2950


## 8. Infer on test

In [ ]:
infer_result = infer_client.run(
    model_address=model_address,
    data_address=test_addr,
    output=f"infer_del_{run_id}",
    dataset_column="smiles",
)
inference_address = infer_result["inference_results_address"]
predictions = data_client.get(inference_address)
print("inference:", inference_address)
print(predictions.head())


inference: deepchem://del_profile/del_project/infer_del_e50147e2.csv
                                                   X   y_preds
0  CNC(=O)[C@H](CCNC(=O)[C@@H](NCc1ccc(-n2ccnc2)c...  0.358057
1  CNC(=O)[C@H](CO)NC(=O)[C@@H]1CCC[C@H](NC(=O)c2...  0.462089
2  CNC(=O)[C@@H](CCNC(=O)[C@@H]1CCC[C@H](NCc2cn(C...  0.359823
3  CNC(=O)[C@@H](CCNC(=O)[C@@H](CNC(=O)C1CCC(C(C)...  0.506775
4  CNC(=O)[C@@H](CCNC(=O)c1ccc(CNC(=O)c2cc3cccc(F...  0.517612


## 9. Non-unified denoise (hit labels)
Z-score enrichment plus binary hits.

In [ ]:
nu_denoise_result = del_denoise_client.run(
    dataset_address=dataset_address,
    output_key=f"del_denoised_nu_{run_id}",
    strategy="non_unified",
    control_cols=["seq_matrix_1", "seq_matrix_2", "seq_matrix_3"],
    target_cols=["seq_target_1", "seq_target_2", "seq_target_3"],
    add_hit_labels=True,
    hit_percentile=90.0,
    drop_duplicates=True,
    use_disynthon_pairs=False,
)
nu_denoised_address = nu_denoise_result["denoised_dataset_address"]
print("non_unified:", nu_denoised_address)

df_nu = data_client.get(nu_denoised_address)
print("shape:", df_nu.shape)
print("target_hits mean:", float(df_nu["target_hits"].mean()))
print("control_hits mean:", float(df_nu["control_hits"].mean()))


non_unified: deepchem://del_profile/del_project/del_denoised_nu_e50147e2.csv
shape: (3111, 20)
target_hits mean: 0.09546769527483125
control_hits mean: 0.07296689167470267


In [17]:
import numpy as np
from scipy.stats import spearmanr

test_dataset = data_client.get(test_addr)
test_y_model = np.asarray(test_dataset.y).reshape(-1)
test_y_pred_model = np.asarray(predictions["y_preds"]).reshape(-1)

pearson_corr = np.corrcoef(test_y_model, test_y_pred_model)[0, 1]
raw_rmse = float(np.sqrt(np.mean((test_y_model - test_y_pred_model) ** 2)))

print("Test RMSE:", round(raw_rmse, 4))
print("Test Pearson r^2:", round(float(pearson_corr) ** 2, 4))

INFO:deepchem.data.datasets:Loading dataset from disk.


Test RMSE: 0.7792
Test Pearson r^2: 0.0646
